# Labeling Enron emails: Request vs Correspondence

**Goal:** get about 500 *Request* and 500 *Correspondence* documents from 517,000 unlabeled Enron emails, without reading them all.

**The idea in one sentence:** a pretrained model suggests a label for every email, and people only *check* a few hundred suggestions instead of labeling from scratch.

| Step | What happens | Who |
|---|---|---|
| 1 | Load the data | notebook |
| 2 | Clean it and throw away junk | notebook |
| 3 | Keyword rules give a first rough label | notebook |
| 4 | A zero-shot model suggests a label + confidence | notebook (GPU) |
| 5 | Keep confident suggestions, export a review sheet | notebook |
| 6 | People confirm or correct the suggestions | **team** |
| 7 | Measure label quality (agreement, model accuracy) | notebook |
| 8 | Build the final train / test files | notebook |
| 9 | *(optional)* Active learning for the next batch | notebook |

**Before you run:** in Kaggle, turn on *Settings → Accelerator → GPU T4* and *Internet on*.

**If an earlier run upgraded packages** (e.g. you see `cannot import name 'httpx'`): use *Run → Factory reset* to restore Kaggle's original packages, then run from the top.

## Step 1: Load the data

The dataset is on Hugging Face (`corbt/enron-emails`): 517,401 emails with columns like `subject`, `from`, `body`. We only need `message_id`, `subject` and `body`.

In [ ]:
# Kaggle already has compatible versions of datasets, transformers and scikit-learn.
# Do NOT upgrade them with pip: mixing versions breaks the imports.
import re
import numpy as np
import pandas as pd
from datasets import load_dataset

raw = load_dataset("corbt/enron-emails", split="train")
df = raw.select_columns(["message_id", "subject", "body"]).to_pandas()
print(df.shape)
df.head(3)

## Step 2: Clean and filter

Most emails are useless for us: replies that quote whole threads, forwards, newsletters, one-line "thanks" messages. We:

1. Cut off quoted/forwarded text (everything after `-----Original Message-----` or `----- Forwarded by`).
2. Remove duplicates (the same email is stored in several folders).
3. Keep emails between 100 and 3,000 characters.
4. Drop obvious automated mail (newsletters, unsubscribe links).
5. Take a random sample of 20,000 to work with. That is plenty; we only need about 1,000 good ones in the end.

In [ ]:
CUT_MARKERS = [
    r"-----\s*Original Message\s*-----",
    r"-{2,}\s*Forwarded by",
    r"\n\s*From:\s.*\n\s*Sent:",
]

def clean_body(text):
    text = str(text)
    for marker in CUT_MARKERS:
        text = re.split(marker, text, maxsplit=1, flags=re.IGNORECASE)[0]
    text = re.sub(r"\s+", " ", text)  # collapse whitespace
    return text.strip()

df["text"] = df["body"].map(clean_body)
df = df.drop_duplicates(subset="text")

length_ok = df["text"].str.len().between(100, 3000)
automated = df["text"].str.contains(r"unsubscribe|newsletter|click here|do not reply", case=False, regex=True)
df = df[length_ok & ~automated]

SAMPLE_SIZE = 20000
work = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42).reset_index(drop=True)
work["input"] = (work["subject"].fillna("") + ". " + work["text"]).str.slice(0, 1500)
print("after filtering:", len(df), "| working sample:", len(work))

## Step 3: Keyword rules (weak supervision)

A **rule** is a tiny function that looks for typical phrases and votes for a label, or abstains. Rules are noisy on their own, but they are instant and they give a second opinion next to the model in Step 4.

- Request phrases: *please send*, *could you*, *can you*, *I need*, *let me know*...
- Correspondence phrases: *FYI*, *attached is*, *for your information*, *here is*...

Feel free to add phrases you notice while reviewing.

In [ ]:
REQUEST_PATTERNS = [
    r"\bplease (send|provide|review|confirm|call|let me know|forward|approve|sign)\b",
    r"\b(could|can|would) you\b",
    r"\bi need\b", r"\bwe need\b",
    r"\bkindly\b", r"\brequest(ing)?\b",
    r"\bas soon as possible\b|\basap\b",
    r"\bby (monday|tuesday|wednesday|thursday|friday|tomorrow|end of day|eod)\b",
]
CORRESPONDENCE_PATTERNS = [
    r"\bfyi\b", r"\bfor your (information|reference|records)\b",
    r"\battached (is|are)\b", r"\bhere (is|are) the\b",
    r"\bjust wanted to let you know\b", r"\bplease (find|see) attached\b",
    r"\bthank(s| you) for\b",
]

def count_hits(text, patterns):
    return sum(bool(re.search(p, text, flags=re.IGNORECASE)) for p in patterns)

def rule_label(text):
    req = count_hits(text, REQUEST_PATTERNS)
    cor = count_hits(text, CORRESPONDENCE_PATTERNS)
    if req > cor:
        return "Request"
    if cor > req:
        return "Correspondence"
    return None  # abstain: no rule fired, or a tie

work["rule_label"] = work["input"].map(rule_label)
print(work["rule_label"].value_counts(dropna=False))

## Step 4: Zero-shot labeling with a pretrained model

**Zero-shot** means the model was never trained on our categories. We give it the email and a few plain-English label descriptions, and it scores how well each description fits. `facebook/bart-large-mnli` was trained to judge whether one sentence follows from another, which is exactly this task.

We give three choices, including **Other**, so the model has somewhere to put newsletters, personal notes and anything that is neither.

**Time:** about 20-40 minutes for 20,000 emails on a T4. Test on 200 first (`TEST_RUN = True`), then set it to `False`.

In [ ]:
import torch
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0 if torch.cuda.is_available() else -1,
)

LABELS = {
    "a request asking the reader to do, send, approve or provide something": "Request",
    "general correspondence that shares information or updates": "Correspondence",
    "a personal, automated or promotional message": "Other",
}
TEMPLATE = "This email is {}."

TEST_RUN = True
batch = work.head(200) if TEST_RUN else work

results = classifier(
    batch["input"].tolist(),
    candidate_labels=list(LABELS.keys()),
    hypothesis_template=TEMPLATE,
    batch_size=16,
)
batch = batch.copy()
batch["zs_label"] = [LABELS[r["labels"][0]] for r in results]
batch["zs_score"] = [r["scores"][0] for r in results]
work = batch.reset_index(drop=True)
work[["input", "rule_label", "zs_label", "zs_score"]].head(10)

## Step 5: Keep confident suggestions and export a review sheet

We trust a suggestion more when:
- the model's top score is high (start with **0.8**), and
- the keyword rule agrees, or stayed silent.

Then we export a **review sheet**: a balanced random sample per label with an empty `final_label` column for people to fill in. We also mark 10% of rows to be labeled by **two** people, to measure agreement in Step 7.

In [ ]:
THRESHOLD = 0.8

rule_ok = work["rule_label"].isna() | (work["rule_label"] == work["zs_label"])
work["confident"] = (work["zs_score"] >= THRESHOLD) & rule_ok & work["zs_label"].isin(["Request", "Correspondence"])
print(work.groupby("zs_label")["confident"].agg(["sum", "count"]))

PER_CLASS = 350  # rows each class gets in the review sheet
parts = []
for label in ["Request", "Correspondence"]:
    pool = work[work["confident"] & (work["zs_label"] == label)]
    parts.append(pool.sample(n=min(PER_CLASS, len(pool)), random_state=1))
# a few low-confidence rows too, so we learn where the model struggles
parts.append(work[~work["confident"]].sample(n=min(100, (~work["confident"]).sum()), random_state=1))

review = pd.concat(parts).sample(frac=1, random_state=2).reset_index(drop=True)
review["suggested_label"] = review["zs_label"]
review["final_label"] = ""        # filled in by the reviewer
review["reviewer"] = (review.index % 6) + 1   # Member 1..6, round robin
review["double_check"] = review.index % 10 == 0  # 10% labeled twice

cols = ["message_id", "subject", "text", "suggested_label", "zs_score", "final_label", "reviewer", "double_check"]
review[cols].to_csv("review_sheet.csv", index=False)
work.to_csv("work_with_suggestions.csv", index=False)
print("review rows:", len(review))

## Step 6: Review the suggestions

Every row in `review_sheet.csv` gets a correct label in `final_label`:

- `Request`: asks the reader to do, send, approve or provide something.
- `Correspondence`: shares information; no action expected. Official notices count here.
- `Other`: calendar entries, automated system mail, newsletters, jokes, personal chat, form-letter petitions.

**In this project the review was done by an LLM (Claude)**, and the `label_source` column records that. Upload the reviewed file here as `review_done.csv` and continue with Step 7.

## Step 7: Check the reviewed labels

Upload the reviewed sheet as `review_done.csv`. In this project the review was done by an LLM (Claude), recorded in the `label_source` column, with no second human labeler, so there is no Cohen's kappa to compute.

What we can measure: **how often the zero-shot model's suggestion was kept by the review.** If confident suggestions are kept less than 90% of the time, unreviewed suggestions are not good enough to train on, and Step 8 leaves them out.

In [ ]:
from sklearn.metrics import classification_report

done = pd.read_csv("review_done.csv")
done = done[done["final_label"].isin(["Request", "Correspondence", "Other"])]
if "label_source" not in done.columns:
    done["label_source"] = "human"

print("label source:", done["label_source"].value_counts().to_dict())
print("reviewed labels:", done["final_label"].value_counts().to_dict())
print(pd.crosstab(done["suggested_label"], done["final_label"], margins=True))

# how good were the model's suggestions?
print(classification_report(done["final_label"], done["suggested_label"], zero_division=0))
confident_reviewed = done[done["zs_score"] >= THRESHOLD]
print("accuracy of confident suggestions (score >= threshold):",
      round((confident_reviewed["final_label"] == confident_reviewed["suggested_label"]).mean(), 3))

## Step 8: Build the final files

- **Test set: reviewed rows only.** If the test set used model suggestions, we would be measuring agreement with the zero-shot model, not real accuracy.
- **Training set:** the remaining reviewed rows, plus confident unreviewed suggestions, but **only if** Step 7 showed the suggestions were at least 90% correct.
- We keep a `label_source` column so the error analysis can compare human vs. automatic labels.

In [ ]:
from sklearn.model_selection import train_test_split

human = done[done["final_label"].isin(["Request", "Correspondence"])].copy()
human["label"] = human["final_label"]  # label_source column comes from review_done.csv

train_h, test = train_test_split(human, test_size=0.3, stratify=human["label"], random_state=42)

# judge only the confident suggestions: those are the ones used without review
conf_rows = done[(done["zs_score"] >= THRESHOLD) & done["suggested_label"].isin(["Request", "Correspondence"])]
suggestion_accuracy = (conf_rows["final_label"] == conf_rows["suggested_label"]).mean()
USE_AUTO = suggestion_accuracy >= 0.90
print("suggestion accuracy:", round(suggestion_accuracy, 3), "| use unreviewed suggestions:", USE_AUTO)

parts = [train_h]
if USE_AUTO:
    auto = work[work["confident"] & ~work["message_id"].isin(done["message_id"])].copy()
    auto["label"] = auto["zs_label"]
    auto["label_source"] = "zero-shot"
    parts.append(auto)
train = pd.concat(parts, ignore_index=True)

keep = ["message_id", "subject", "text", "label", "label_source"]
train[keep].to_csv("enron_train.csv", index=False)
test[keep].to_csv("enron_test.csv", index=False)
print("train:", train["label"].value_counts().to_dict())
print("test:", test["label"].value_counts().to_dict())

## Step 9 (optional): Active learning for the next batch

If one class is still short, don't review random rows. Train a quick model on what people confirmed, and let it pick the emails it is **least sure about**. Those rows teach the next model the most per minute of human time.

Repeat Steps 6-9 with the new sheet until each class has enough rows.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000)
X_lab = vec.fit_transform(human["text"])
clf = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_lab, human["label"])

unlabeled = work[~work["message_id"].isin(done["message_id"])].copy()
proba = clf.predict_proba(vec.transform(unlabeled["text"]))
unlabeled["margin"] = np.abs(proba[:, 0] - proba[:, 1])  # small margin = model is unsure
unlabeled["suggested_label"] = clf.classes_[proba.argmax(axis=1)]

next_batch = unlabeled.nsmallest(200, "margin")
next_batch["final_label"] = ""
next_batch[["message_id", "subject", "text", "suggested_label", "final_label"]].to_csv("review_sheet_round2.csv", index=False)
print("next review batch:", len(next_batch))

## What to write in the data card

- Source: Enron email corpus (cleaned version on Hugging Face), N emails after filtering.
- Method: keyword rules + zero-shot suggestions (`facebook/bart-large-mnli`, threshold 0.8), then every suggestion reviewed by an LLM (Claude) following the labeling guideline (`label_source` column).
- Quality: share of zero-shot suggestions kept by the review (Step 7). No inter-annotator agreement was measured, because there was no second independent labeler.
- Test set: reviewed labels only; never unreviewed zero-shot suggestions.